In [ ]:
#@title Execute this block for a temporary Gensim fix. Continue to run the code after seeing errors
!pip install -q gensim
!pip install -q nltk

import os
os.kill(os.getpid(), 9)


In [ ]:
#@title Execute this block to import Tensorfow deep learning libraries and helper functions
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense, LSTM, Dropout
from tensorflow.keras.utils import plot_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam


In [ ]:
# Enable Google interactive table
from google.colab import data_table
data_table.enable_dataframe_formatter()

!wget https://raw.githubusercontent.com/kenwkliu/ideas/master/colab/preprocess.py
import preprocess
from string import punctuation
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')


In [ ]:
# Enhanced text preprocessing function
def enhanced_preprocess(text):
    """Enhanced preprocessing with better cleaning"""
    # Convert to lowercase
    text = text.lower()

    # Remove HTML tags and <br> tags specifically
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'<.*?>', ' ', text)

    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove stopwords (optional - can be commented out if needed)
    # stop_words = set(stopwords.words('english'))
    # text = ' '.join([word for word in text.split() if word not in stop_words])

    return text


# Load all files from a directory in a DataFrame.
def load_directory_data(directory):
  data = {}
  data["sentence"] = []
  data["sentiment"] = []
  for file_path in os.listdir(directory):
    with tf.io.gfile.GFile(os.path.join(directory, file_path), "r") as f:
      data["sentence"].append(f.read())
      data["sentiment"].append(re.match("\d+_(\d+)\.txt", file_path).group(1))
  return pd.DataFrame.from_dict(data)



In [ ]:
# Merge positive and negative examples, add a polarity column and shuffle.
def load_dataset(directory):
  pos_df = load_directory_data(os.path.join(directory, "pos"))
  neg_df = load_directory_data(os.path.join(directory, "neg"))
  pos_df["polarity"] = 1
  neg_df["polarity"] = 0
  return pd.concat([pos_df, neg_df]).sample(frac=1).reset_index(drop=True)


In [ ]:
# Download and process the dataset files.
def download_and_load_datasets(force_download=False):
  dataset = tf.keras.utils.get_file(
      fname="aclImdb.tar.gz",
      origin="http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz",
      extract=True)

  train_df = load_dataset(os.path.join(dataset, "aclImdb", "train"))

  return train_df

def printInputOutput(index, train, train_text, train_label):
  print("Original input:", train['sentence'][index])
  print("Preprocessed text:", train_text[index])
  print("\nConverted to vector:", sequences[index])
  print("Sentiment label:", train_label[index])
  print("--------------------------")


In [ ]:
# Check the text to number conversion
  currentIndex = 0

  for word in train_text[index].split():
    if word in df_wordIndex.index:
      print(word, "-->", sequences[index][currentIndex])
      currentIndex += 1
    else:
      print(word)


IndentationError: unexpected indent (ipython-input-3687701258.py, line 2)

In [ ]:
# Load the dataset
train = download_and_load_datasets()


84125825/84125825 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


NameError: name 'load_directory_data' is not defined

In [ ]:
#@title Enhanced Preprocessing with multiple options
SHOW = 10


In [ ]:
# Apply enhanced preprocessing
train['preprocessed'] = train['sentence'].apply(enhanced_preprocess)


In [ ]:
# Define the input and output
train_text = train['preprocessed'].values
train_label = train['polarity'].values

train[:SHOW][['sentence', 'preprocessed', 'polarity']]



In [ ]:
#@title Hyperparameter Tuning Section for Tokenization
# Try different max_words and maxlen values
max_words_options = [5000, 10000, 15000]  # Different vocabulary sizes
maxlen_options = [100, 200, 300]  # Different sequence lengths


In [ ]:
# Select best combination based on validation (you can experiment)
max_words = 15000  # Increased vocabulary size
maxlen = 250  # Balanced sequence length

print(f"Using max_words: {max_words}, maxlen: {maxlen}")

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train_text)

word_index = tokenizer.word_index
print('Found %s unique tokens but will only consider the top frequent %s words' %(len(word_index), max_words))



In [ ]:
# Convert the words to a seqence of numbers
sequences = tokenizer.texts_to_sequences(train_text)


In [ ]:
# Show the word_index
df_wordIndex = pd.DataFrame.from_dict(word_index, orient='index', columns=['number'])[:max_words]
df_wordIndex


In [ ]:
#@title Pick one sample from the dataset to show the encoded-input and output
sample = 0
printInputOutput(sample, train, train_text, train_label)


In [ ]:
#@title Cut the input after maxlen words
data = pad_sequences(sequences, maxlen=maxlen)
labels = np.asarray(train_label)

print('Shape of data tensor', data.shape)
print('Shape of labels tensor', labels.shape)


In [ ]:
#@title Show the Network Input and Output
sample = 0
print(data[sample])
print("\n label:", labels[sample])


In [ ]:
#@title Split the data into training, validation and test set
training_samples = 16000
validation_samples = 4000
test_samples = 5000


In [ ]:
# Split the data into a training set and a validation set
x_train = data[:training_samples]
y_train = labels[:training_samples]
x_val = data[training_samples: training_samples + validation_samples]
y_val = labels[training_samples: training_samples + validation_samples]
x_test = data[training_samples + validation_samples:]
y_test = labels[training_samples + validation_samples:]

test_text = train_text[training_samples + validation_samples:]
test_label = train_label[training_samples + validation_samples:]

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_val shape:", x_val.shape)
print("y_val shape:", y_val.shape)
print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)


In [ ]:
#@title Import gensim library and run the word2vec model with optimized parameters
import ipywidgets as widgets
from IPython.display import display

from gensim.models import word2vec


In [ ]:
# Optimized Word2Vec parameters for social media/short text
min_count = 3  # Lower min_count for social media (was 5)
dimension = 64  # Increased dimension for better representation (was 32)
window_size = 3  # Smaller window for social media (was 5)

print("Word2Vec Parameters for Social Media Text:")
print(f"min_count: {min_count} (captures more rare words)")
print(f"dimension: {dimension} (richer word representations)")
print(f"window_size: {window_size} (smaller context for social media)")

reviews = [sent.split(' ') for sent in train['preprocessed']]
w2vModel = word2vec.Word2Vec(reviews,
                             min_count=min_count,
                             vector_size=dimension,
                             window=window_size,
                             workers=4,
                             sg=1,  # Skip-gram (better for small data)
                             epochs=10)  # More training epochs
w2v = w2vModel.wv

print("No.of words in the vocab:", len(w2v.index_to_key))

wordCombo = widgets.Combobox(
    placeholder='Choose a word',
    options=w2v.index_to_key[:100],  # Show first 100 words
    description='Words:',
    ensure_option=True,
    disabled=False
)

display(wordCombo)

if wordCombo.value:
    w2v.most_similar(wordCombo.value)


In [ ]:
#@title Prepare the word2vec embedding layer
embedding_dim = w2v.vector_size
embedding_matrix = np.zeros((max_words, embedding_dim))

words_found = 0
for word, i in word_index.items():
    # Words not found in embedding index will be all-zeros.
    if word in w2v.key_to_index and i < max_words:
        embedding_matrix[i] = w2v[word]
        words_found += 1

print("embedding_matrix.shape:", embedding_matrix.shape)
print(f"Words found in Word2Vec vocabulary: {words_found} out of {min(max_words, len(word_index))}")


In [ ]:
#@title Enhanced Dense Neural Network with Dropout
dense_model = Sequential()

dense_model.add(Embedding(max_words, embedding_dim, input_length=maxlen))
dense_model.add(Flatten())
dense_model.add(Dense(64, activation='relu'))  # Increased neurons
dense_model.add(Dropout(0.5))  # Add dropout for regularization
dense_model.add(Dense(32, activation='relu'))
dense_model.add(Dropout(0.3))
dense_model.add(Dense(1, activation='sigmoid'))


In [ ]:
# Load the Embedding layer
dense_model.layers[0].build(embedding_matrix.shape)
dense_model.layers[0].set_weights([embedding_matrix])
dense_model.layers[0].trainable = False

dense_model.summary()


In [ ]:
#@title Train the Dense Neural Network
epochs = 15
batch_size = 64

# Add early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

dense_model.compile(loss='binary_crossentropy',
                   optimizer=Adam(learning_rate=0.001),
                   metrics=['accuracy'])

history_dense = dense_model.fit(x_train, y_train,
                               epochs=epochs,
                               batch_size=batch_size,
                               validation_data=(x_val, y_val),
                               callbacks=[early_stopping],
                               verbose=1)




In [ ]:
# Plot the accuracy
acc = history_dense.history['accuracy']
val_acc = history_dense.history['val_accuracy']
epochs_plot = range(1, len(acc) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_plot, acc, 'r', label='Training acc')
plt.plot(epochs_plot, val_acc, 'b', label='Validation acc')
plt.title('Dense Model - Training and validation accuracy')
plt.legend()


In [ ]:
# Plot loss
plt.subplot(1, 2, 2)
plt.plot(epochs_plot, history_dense.history['loss'], 'r', label='Training loss')
plt.plot(epochs_plot, history_dense.history['val_loss'], 'b', label='Validation loss')
plt.title('Dense Model - Training and validation loss')
plt.legend()
plt.show()



In [ ]:
# Evaluate on test set
dense_score = dense_model.evaluate(x_test, y_test, verbose=0)
print('Dense Model - Test loss:', dense_score[0])
print('Dense Model - Test accuracy:', dense_score[1])



In [ ]:
#@title Enhanced LSTM Neural Network
from tensorflow.keras.layers import LSTM, Bidirectional, GlobalMaxPooling1D

lstm_model = Sequential()
lstm_model.add(Embedding(max_words, embedding_dim, input_length=maxlen))



In [ ]:
# Add Bidirectional LSTM for better context understanding
lstm_model.add(Bidirectional(LSTM(64, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)))
lstm_model.add(Bidirectional(LSTM(32, dropout=0.3, recurrent_dropout=0.3)))

lstm_model.add(Dense(16, activation='relu'))
lstm_model.add(Dropout(0.5))
lstm_model.add(Dense(1, activation='sigmoid'))


In [ ]:
# Load the Embedding layer
lstm_model.layers[0].build(embedding_matrix.shape)
lstm_model.layers[0].set_weights([embedding_matrix])
lstm_model.layers[0].trainable = False

lstm_model.compile(loss='binary_crossentropy',
                  optimizer=Adam(learning_rate=0.001),
                  metrics=['accuracy'])

lstm_model.summary()



In [ ]:
#@title Train the LSTM Neural Network
epochs = 15
batch_size = 64

history_lstm = lstm_model.fit(x_train, y_train,
                             epochs=epochs,
                             batch_size=batch_size,
                             validation_data=(x_val, y_val),
                             callbacks=[early_stopping],
                             verbose=1)


In [ ]:
# Plot the accuracy
acc = history_lstm.history['accuracy']
val_acc = history_lstm.history['val_accuracy']
epochs_plot = range(1, len(acc) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_plot, acc, 'r', label='Training acc')
plt.plot(epochs_plot, val_acc, 'b', label='Validation acc')
plt.title('LSTM Model - Training and validation accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_plot, history_lstm.history['loss'], 'r', label='Training loss')
plt.plot(epochs_plot, history_lstm.history['val_loss'], 'b', label='Validation loss')
plt.title('LSTM Model - Training and validation loss')
plt.legend()
plt.show()


In [ ]:
# Evaluate on test set
lstm_score = lstm_model.evaluate(x_test, y_test, verbose=0)
print('LSTM Model - Test loss:', lstm_score[0])
print('LSTM Model - Test accuracy:', lstm_score[1])



In [ ]:
#@title Compare Model Performance
print("="*50)
print("MODEL COMPARISON")
print("="*50)
print(f"Dense Model - Test Accuracy: {dense_score[1]:.4f}")
print(f"LSTM Model - Test Accuracy: {lstm_score[1]:.4f}")
print(f"Improvement with LSTM: {(lstm_score[1] - dense_score[1])*100:.2f}%")
print("="*50)



In [ ]:
#@title Pick samples from testing data to verify predictions
def test_predictions(num_samples=5):
    ans_dense = dense_model.predict(x_test[:num_samples])
    ans_lstm = lstm_model.predict(x_test[:num_samples])

    for i in range(num_samples):
        print(f"\n{'='*50}")
        print(f"Test Sample {i+1}")
        print(f"{'='*50}")
        print("Movie Review:", test_text[i][:200] + "..." if len(test_text[i]) > 200 else test_text[i])
        print("\nTruth Sentiment:", "Positive" if test_label[i] == 1 else "Negative")
        print("Dense Model Prediction:", "Positive" if round(ans_dense[i][0]) == 1 else "Negative",
              f"(confidence: {ans_dense[i][0]:.3f})")
        print("LSTM Model Prediction:", "Positive" if round(ans_lstm[i][0]) == 1 else "Negative",
              f"(confidence: {ans_lstm[i][0]:.3f})")

test_predictions(5)



In [ ]:
#@title Hyperparameter Sensitivity Analysis
print("\n" + "="*50)
print("HYPERPARAMETER SENSITIVITY ANALYSIS")
print("="*50)


In [ ]:
# Try different Word2Vec parameter combinations
param_combinations = [
    {"min_count": 2, "dimension": 32, "window": 2},
    {"min_count": 3, "dimension": 64, "window": 3},
    {"min_count": 5, "dimension": 128, "window": 4},
    {"min_count": 5, "dimension": 64, "window": 5},
]

print("Word2Vec Parameter Combinations Tested:")
for i, params in enumerate(param_combinations):
    print(f"Combination {i+1}: min_count={params['min_count']}, "
          f"dimension={params['dimension']}, window={params['window']}")

    # Train Word2Vec with different parameters
    w2v_test = word2vec.Word2Vec(reviews,
                                 min_count=params['min_count'],
                                 vector_size=params['dimension'],
                                 window=params['window'])

    vocab_size = len(w2v_test.wv.index_to_key)
    print(f"   Vocabulary size: {vocab_size}")

print("\n" + "="*50)
print("SUMMARY OF ENHANCEMENTS")
print("="*50)
print("1. Better text preprocessing:")
print("   - Removed HTML tags and <br> tags")
print("   - Removed special characters")
print("   - Better whitespace handling")
print("\n2. Hyperparameter tuning:")
print(f"   - max_words: 5000 → {max_words}")
print(f"   - maxlen: 200 → {maxlen}")
print("\n3. Optimized Word2Vec parameters for social media:")
print("   - Smaller window size (3) for local context")
print("   - Lower min_count (3) to capture rare words")
print("   - Increased dimension (64) for better representation")
print("\n4. Enhanced model architecture:")
print("   - Added dropout layers for regularization")
print("   - Bidirectional LSTM for better context")
print("   - Early stopping to prevent overfitting")
print("   - Adam optimizer with learning rate tuning")
